In [13]:
from asyncio import start_server
from calendar import weekday
from select import KQ_NOTE_WRITE

import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import altair as alt
import plotly.express as px
import plotly.graph_objects as go
import great_tables as tg
import datetime as dt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from urllib3.util import wait_for_write

In [14]:
df_path = r'/Users/zygimantas/Downloads/archive (1)/fitness_dataset.csv'

In [15]:
df = pl.read_csv(df_path)

In [16]:
df

age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
i64,i64,i64,f64,f64,f64,f64,f64,str,str,i64
56,152,65,69.6,117.0,null,2.37,3.97,"""no""","""F""",1
69,186,95,60.8,114.8,7.5,8.77,3.19,"""0""","""F""",1
46,192,103,61.4,116.4,null,8.2,2.03,"""0""","""F""",0
32,189,83,60.2,130.1,7.0,6.18,3.68,"""0""","""M""",1
60,175,99,58.1,115.8,8.0,9.95,4.83,"""yes""","""F""",1
…,…,…,…,…,…,…,…,…,…,…
52,173,98,60.7,106.1,null,1.54,3.25,"""1""","""M""",1
61,186,74,51.4,123.8,9.4,8.63,3.15,"""no""","""M""",1
77,198,89,76.7,103.6,8.3,1.98,3.36,"""yes""","""M""",0


In [17]:
df = df.with_columns(
    cs.integer().shrink_dtype()
)

In [19]:
df.null_count()

age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,160,0,0,0,0,0


In [22]:
df = df.with_columns(
    pl.col('sleep_hours').fill_null(0.0)
)

In [23]:
df.describe()

statistic,age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64
"""count""",2000.0,2000.0,2000.0,2000.0,2000.0,2000.0,2000.0,2000.0,"""2000""","""2000""",2000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""",0.0
"""mean""",49.114,174.533,83.5405,70.2886,119.90885,6.91225,5.03514,2.99904,null,null,0.3995
"""std""",17.926564,14.37175,25.852534,11.846339,14.578032,2.49646,2.864156,1.136383,null,null,0.489918
"""min""",18.0,150.0,30.0,45.0,90.0,0.0,0.0,1.0,"""0""","""F""",0.0
"""25%""",34.0,162.0,64.0,62.1,109.7,6.1,2.55,2.04,null,null,0.0
"""50%""",49.0,174.0,83.0,70.3,120.0,7.4,5.07,2.98,null,null,0.0
"""75%""",65.0,187.0,102.0,78.4,129.8,8.4,7.47,3.95,null,null,1.0
"""max""",79.0,199.0,250.0,118.6,171.2,12.0,10.0,4.99,"""yes""","""M""",1.0


In [25]:
X = df.select(
    cs.exclude('is_fit')
)

In [27]:
y = df.get_column(
    'is_fit'
)

In [30]:
X_numeric = df.select(
    cs.numeric()
)

X_categorical = df.select(
    cs.string()
)

In [34]:
X_categorical.select(
    pl.all().n_unique()
)

smokes,gender
u32,u32
4,2


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [37]:
column_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(
            handle_unknown='ignore', sparse_output=False, drop='first'),
        X.select(cs.string()).columns),
        ('passthrough', 'passthrough', X.select(cs.numeric()).columns)
    ]
)

In [38]:
column_transformer

,transformers,"[('encoder', ...), ('passthrough', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


In [39]:
X_encoded = column_transformer.fit_transform(X)

In [45]:
feature_names = column_transformer.named_transformers_['encoder'].get_feature_names_out()

In [47]:
numeric_columns = X.select(cs.numeric()).columns

In [49]:
all_feature_names = list(feature_names) + list(numeric_columns)

In [50]:
all_feature_names

['smokes_1',
 'smokes_no',
 'smokes_yes',
 'gender_M',
 'age',
 'height_cm',
 'weight_kg',
 'heart_rate',
 'blood_pressure',
 'sleep_hours',
 'nutrition_quality',
 'activity_index']

In [55]:
X_encoded = pl.DataFrame(
    X_encoded, schema=all_feature_names
)

In [56]:
from sklearn.model_selection import train_test_split

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [58]:
from sklearn.preprocessing import StandardScaler

In [59]:
scaler = StandardScaler()

In [60]:
scaler

,copy,True
,with_mean,True
,with_std,True
